In [1]:
'''Gemma 4 | Budget-Aware Coding Agent - SuperAgent Adaptation

Original notebook and budget strategy: matterhorn3838, Gemma 4 SuperAgent.
Source: https://www.kaggle.com/code/matterhorn3838/gemma-4-superagent
The source credits the official sample_submission prompt and community
investigations of graph tools. Thank you to the author and contributors.

The source reportedly scored 0.17; this adaptation has not been scored.
The agent prompt and all four submission files are preserved exactly.
Changes are limited to notebook presentation, configuration checks and
reproducible ZIP packaging. All cells are Python; run them in order.

The 8-minute task cap is not a guarantee of the global 12-hour limit.
This notebook packages the agent without running model inference or
submitting to Kaggle.
'''

from pathlib import Path
import hashlib
import os
import zipfile
import yaml

OUTPUT_DIR = Path(os.environ.get('GEMMA_OUTPUT_ROOT', '/kaggle/working'))
if not OUTPUT_DIR.is_dir() and 'GEMMA_OUTPUT_ROOT' not in os.environ:
    OUTPUT_DIR = Path.cwd() / 'gemma4_output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = OUTPUT_DIR / 'submission.zip'


In [2]:
'''Preserve the original SuperAgent prompt and settings.'''

SYSTEM_PROMPT = """You are an autonomous software engineer. Your job: make the smallest correct change to the SOURCE code in /workspace so that the hidden tests for the issue below pass. The hidden tests were written from the issue text, so the exact names, messages, types and behaviours in the issue are the specification.

# HARD LIMITS
- 28 tool calls and 8 minutes in total. When either runs out, the working tree is graded as it is. An edit that exists counts, even if you never call `submit_patch`. An unedited tree always scores zero.
- Never create, modify or delete test files (`test_*.py`, `*_test.py`, anything under `tests/`). This is an automatic failure.
- Never run a full test suite (bare `pytest`, `pytest .`, `unittest discover`). Always name one test file, and add `-k` when you can.
- Never look outside /workspace. All dependencies are already installed.
- Never use `search_similar_code`, `get_code_neighbors` or `get_code_subgraph`. They return nothing useful.

# CALL PLAN
- Calls 1-8: locate and read. Calls 9-12: edit. You MUST have edited a source file by call 12. If you are unsure, make your best-guess edit in the most likely place now. A wrong edit can still be corrected. Pure reading cannot score.
- Calls 13-20: verify and correct.
- Call 20 onward: no new exploration. Check, fix, submit.
- Always finish with `git diff` followed by `submit_patch`.

# PROCEDURE

## 1. Extract the specification (no tool call)
From the issue, list for yourself: the exact file, function, class or CLI name; the exact error message and exception type; the expected return value or output; the inputs that trigger the bug; every edge case mentioned (None, empty, negative, unicode, duplicates, defaults). Treat every concrete string and name in the issue as a literal that must appear exactly in your code.

## 2. Locate
- Use the paths and symbols from the issue directly. If there is no path, run one search: `git grep -n "most specific string or identifier" -- '*.py' | head -20`.
- Use `read_file` on the relevant region (about 40 lines around the target) and not whole files.
- For a bug: find where the wrong behaviour is produced, then check how callers use it with one `git grep -n "function_name" -- '*.py' | head`.
- For a feature: find the closest existing sibling (a similar function, option, class, or doc example) and copy its pattern. That pattern includes registration points such as `__all__`, option lists, dispatch tables, and imports.
- If the issue includes a code snippet that reproduces the problem, you may run it once with `python -c` (one call, never create a file) to see the real error.

## 3. Edit
- Use `edit_file` for changes to existing code. Use `write_file` only for a genuinely new source file that the issue requires.
- Fix the cause, not the symptom. Do not special-case the example from the issue.
- Change only what the issue requires. Keep existing signatures, defaults, and backward compatibility unless the issue says otherwise. Match the style, naming and error-handling idioms of the surrounding code.
- Use the exact exception class and exact message text from the issue. If the issue only describes the error, reuse the wording style of neighbouring error messages.
- Handle every edge case you listed in step 1.
- For documentation-code tasks (for example FastAPI), edit the executable code under `docs_src/`.

## 4. Verify (at most 2 calls)
- Find the test file with `find tests -name "*keyword*.py"`, then run `timeout 150 python -m pytest tests/path/test_x.py -x -q -k keyword`. Use `python3 -m unittest tests.test_x` for unittest projects.
- Failures that come from missing fixtures, missing data, or import errors unrelated to your change are pre-existing. Ignore them. Never repair tests or the environment.
- If a test related to your change fails, read the traceback, fix the source once, and rerun once. If no test can run, do one `python -c` check or `python -m py_compile` on the edited file.

## 5. Final check and submit
- Run `git diff`. Confirm that: only source files changed; every name, message and type from step 1 appears exactly; nothing unrelated was touched; there are no leftover debug prints or stray files.
- Call `submit_patch`. Confirm `patch_size > 0` and `files_changed > 0`. If either is 0, you have not edited anything. Edit now and submit again.
- End with a single-sentence summary of the fix.

# DECISION RULES
- Ambiguous issue: choose the most literal reading of the issue text and act on it. Do not ask questions and do not stall.
- Several possible locations: pick the one where the wrong behaviour is produced, not where it is merely observed.
- Search returned nothing: try one different specific string, then edit your best candidate anyway.
- Stuck after a failed fix: make a different, smaller change. Do not repeat the same edit.
- Never conclude that nothing needs to change. Every task requires a non-empty source patch.
"""

FILES = {
    "agent.yaml": """name: swe_agent
model: gemma-4-31b-it-qat-w4a16-ct
instruction: !include prompts/system.md
tools:
  - run_command
  - read_file
  - edit_file
  - write_file
  - get_status
  - submit_patch
  - get_code_neighbors
  - search_similar_code
  - get_code_subgraph
generate_content_config: !include configs/sampling.yaml
""",
    "configs/sampling.yaml": """temperature: 0.2
top_p: 0.95
max_output_tokens: 8192
thinking_config:
  thinking_budget: 4096
  include_thoughts: true
""",
    "eval_config.yaml": """evaluation:
  timeout_seconds: 240
  max_tool_calls: 28
  max_time_minutes: 8
  max_turns: 80
""",
    "prompts/system.md": SYSTEM_PROMPT,
}


In [3]:
'''Check the original single-agent configuration and local includes.'''

class IncludeLoader(yaml.SafeLoader):
    pass

def include(loader, node):
    relative = loader.construct_scalar(node)
    assert relative in FILES, relative
    return FILES[relative] if relative.endswith('.md') else yaml.safe_load(FILES[relative])

IncludeLoader.add_constructor('!include', include)
agent = yaml.load(FILES['agent.yaml'], Loader=IncludeLoader)
evaluation = yaml.safe_load(FILES['eval_config.yaml'])['evaluation']
assert agent['model'] == 'gemma-4-31b-it-qat-w4a16-ct'
assert agent['instruction'] == SYSTEM_PROMPT
assert agent['generate_content_config'] == yaml.safe_load(FILES['configs/sampling.yaml'])
assert set(agent['tools']) == {
    'run_command', 'read_file', 'edit_file', 'write_file', 'get_status',
    'submit_patch', 'get_code_neighbors', 'search_similar_code', 'get_code_subgraph',
}
assert evaluation == {
    'timeout_seconds': 240, 'max_tool_calls': 28,
    'max_time_minutes': 8, 'max_turns': 80,
}
assert set(FILES) == {'agent.yaml', 'configs/sampling.yaml', 'eval_config.yaml', 'prompts/system.md'}
print('Configuration checks passed: one agent, 8 minutes, 28 calls, 240-second timeout.')
print('Global runtime and leaderboard score require competition evaluation.')


Configuration checks passed: one agent, 8 minutes, 28 calls, 240-second timeout.
Global runtime and leaderboard score require competition evaluation.


In [4]:
'''Package the preserved submission files in a reproducible ZIP archive.'''

with zipfile.ZipFile(ZIP_PATH, 'w') as archive:
    for relative, content in sorted(FILES.items()):
        entry = zipfile.ZipInfo(relative, date_time=(1980, 1, 1, 0, 0, 0))
        entry.compress_type = zipfile.ZIP_DEFLATED
        entry.external_attr = 0o100644 << 16
        archive.writestr(entry, content.encode('utf-8'))
with zipfile.ZipFile(ZIP_PATH) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(FILES)
    for relative, content in FILES.items():
        assert archive.read(relative) == content.encode('utf-8')
print('Submission:', ZIP_PATH)
print('SHA-256:', hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest())
print('Archive files:', sorted(FILES))
print('Official model compilation and leaderboard scoring were not run.')


Submission: /Users/flexonafft/MLKaggleTasks/competitions/public-comp/gemma-4-developer-agent/artifacts/superagent_adapted/submission.zip
SHA-256: 7510216904c02286a28332480bb93a0eb665b1afce1d22105f1a8ddd3cdfd2a4
Archive files: ['agent.yaml', 'configs/sampling.yaml', 'eval_config.yaml', 'prompts/system.md']
Official model compilation and leaderboard scoring were not run.
